In [27]:
import os, json
from pathlib import Path
import fitz
import numpy as np
import cv2
from PIL import Image

def build_packs_for_path(
    input_path,  # PDF ou dossier
    out_root="packs_out",
    recursive=False,

    zoom=2.0,
    max_pages=None,  # None = toutes les pages

    #  Filtrage / fusion (général) 
    text_gate_chars=900,  # Page très textuelle : pas de découpe image en mode pixel
    min_xref_area_pt2=2000,  # Ignore les petites images (icônes, bruit)
    merge_gap_pt=25,  # Fusionne des morceaux d'une même figure (écart toléré)
    margin_pt=12,   # Petite marge autour du cadre (évite de couper des traits)

    # Extension contrôlée (important) 
    expand_candidates_px=(80, 160, 260),  # Fenêtres d’extension candidates (petit = plus précis)
    max_expand_factor=2.2,  # Limite l’agrandissement : évite d’aspirer le texte
    area_pen_mult=10.0,  # Pénalité sur les cadres trop grands
    small_region_area=0.03,   # Trop petit : probablement un bout (ex : coin)
    huge_region_area=0.75,    # Trop grand : risque de prendre une demi-page de texte

    #  "Ça ressemble à du texte" (important) 
    small_cc_area=220,   # Composant petit (aire) => typique des lettres
    small_cc_h=42,    # Composant petit (hauteur) => typique des lettres
    text_pen_mult=0.35,  # Pénalité si la zone est très "textuelle"

    #  QR (option) 
    enable_qr=True,
    qr_expand=(0.12, 0.12, 0.18, 0.12), # Extension relative autour du QR (gauche, droite, haut, bas)

    #  Secours 
    pixel_fallback=True, # Pas d’image embarquée : on tente via l’image rendue (utile pour les vectoriels)
    save_text_pages=False, # Si True : on sauvegarde aussi les pages texte en image
    save_full_page_if_empty=False
):
    # Réduit le bruit MuPDF dans la console
    fitz.TOOLS.mupdf_display_errors(False)
    fitz.TOOLS.mupdf_display_warnings(False)

    input_path = Path(input_path)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    # Liste des PDF à traiter
    if input_path.is_dir():
        pdfs = sorted(input_path.rglob("*.pdf")) if recursive else sorted(input_path.glob("*.pdf"))
    else:
        pdfs = [input_path]

    qr = cv2.QRCodeDetector()
    index = []

    for pdf_path in pdfs:
        if not pdf_path.exists():
            continue

        stem = pdf_path.stem
        img_dir = out_root / f"{stem}_images"
        img_dir.mkdir(parents=True, exist_ok=True)
        out_json = out_root / f"{stem}.json"

        doc = fitz.open(str(pdf_path))
        total = len(doc) if max_pages is None else min(len(doc), max_pages)

        uid = 0
        pages_out = []

        for i in range(total):
            page = doc[i]
            page_no = i + 1
            page_w_pt, page_h_pt = float(page.rect.width), float(page.rect.height)

            # Rend la page en image (point d'entrée commun : raster + vectoriel)
            pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

            # Prépare une carte binaire : blanc = fond, noir = "encre"
            arr = np.array(img)
            gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            _, bw = cv2.threshold(gray, 245, 255, cv2.THRESH_BINARY_INV)
            H, W = bw.shape

            # Texte natif du PDF (sans OCR), sert au tri des pages
            text_len = 0
            try:
                t = page.get_text("text")
                if t:
                    text_len = len(t.strip())
            except Exception:
                text_len = 0

            page_item = {"page": page_no, "blocks": []}

            # QR : on garde une zone "QR + contexte"
            if enable_qr:
                qr_data, qr_pts, _ = qr.detectAndDecode(arr)
                if qr_pts is not None and len(qr_pts) > 0:
                    pts = qr_pts.reshape(-1, 2)
                    x1 = int(np.min(pts[:, 0])); y1 = int(np.min(pts[:, 1]))
                    x2 = int(np.max(pts[:, 0])); y2 = int(np.max(pts[:, 1]))
                    exL = int(qr_expand[0] * W); exR = int(qr_expand[1] * W)
                    exU = int(qr_expand[2] * H); exD = int(qr_expand[3] * H)
                    x1 = max(0, x1 - exL); x2 = min(W, x2 + exR)
                    y1 = max(0, y1 - exU); y2 = min(H, y2 + exD)

                    crop = img.crop((x1, y1, x2, y2))
                    name = f"p{page_no:04d}_qr_{uid:06d}.webp"
                    crop.save(str(img_dir / name), "WEBP", quality=85, method=6)
                    page_item["blocks"].append({
                        "type": "qr_context",
                        "path": f"{img_dir.name}/{name}",
                        "bbox_px": [x1, y1, x2, y2]
                    })
                    uid += 1

            # Score d’un cadre : on veut une zone plutôt "graphique" et pas trop grande
            def score_bbox(x1, y1, x2, y2):
                x1 = max(0, x1); y1 = max(0, y1)
                x2 = min(W, x2); y2 = min(H, y2)
                if x2 - x1 < 10 or y2 - y1 < 10:
                    return -1e9, 1.0, 1.0, 1e9

                area_ratio = (x2 - x1) * (y2 - y1) / float(W * H)

                sub = bw[y1:y2, x1:x2]
                ink_ratio = float(np.count_nonzero(sub)) / float(sub.size + 1)

                # Beaucoup de petits composants => souvent du texte
                nlabels, labels, stats, _ = cv2.connectedComponentsWithStats(sub, connectivity=8)
                if nlabels <= 1:
                    small_cc = 0
                else:
                    A = stats[1:, cv2.CC_STAT_AREA]
                    Hh = stats[1:, cv2.CC_STAT_HEIGHT]
                    small_cc = int(np.sum((A < small_cc_area) & (Hh < small_cc_h)))

                # Pénalités de taille
                area_pen = 0.0
                if area_ratio < small_region_area:
                    area_pen += (small_region_area - area_ratio) * area_pen_mult
                if area_ratio > huge_region_area:
                    area_pen += (area_ratio - huge_region_area) * area_pen_mult

                # Pénalité "trop textuel"
                text_pen = min(1.0, small_cc / 2500.0) * text_pen_mult

                score = ink_ratio - area_pen - text_pen
                return score, area_ratio, ink_ratio, small_cc

            # A Images embarquées (xref) : positions des bitmaps dans la page
            rects = []
            try:
                for it in page.get_images(full=True):
                    xref = it[0]
                    for r in page.get_image_rects(xref):
                        if (r.width * r.height) > min_xref_area_pt2:
                            rects.append(r)
            except Exception:
                rects = []

            # Fusion des cadres proches (figure découpée en plusieurs morceaux)
            merged = []
            if rects:
                rects = sorted(rects, key=lambda r: (r.y0, r.x0))
                for r in rects:
                    placed = False
                    for k in range(len(merged)):
                        m = merged[k]
                        expand_m = fitz.Rect(m.x0-merge_gap_pt, m.y0-merge_gap_pt,
                                             m.x1+merge_gap_pt, m.y1+merge_gap_pt)
                        if expand_m.intersects(r):
                            merged[k] = m | r
                            placed = True
                            break
                    if not placed:
                        merged.append(r)

            # Pour chaque zone : plusieurs candidats d’extension, on garde le meilleur
            if merged:
                for r in merged:
                    r2 = fitz.Rect(
                        max(0, r.x0 - margin_pt),
                        max(0, r.y0 - margin_pt),
                        min(page_w_pt, r.x1 + margin_pt),
                        min(page_h_pt, r.y1 + margin_pt),
                    )
                    bx1 = max(0, int(r2.x0 * zoom)); by1 = max(0, int(r2.y0 * zoom))
                    bx2 = min(W, int(r2.x1 * zoom)); by2 = min(H, int(r2.y1 * zoom))
                    if bx2 - bx1 < 10 or by2 - by1 < 10:
                        continue

                    base_area = max(1, (bx2 - bx1) * (by2 - by1))

                    best = None
                    cand_list = [(0, bx1, by1, bx2, by2)]  # sans extension

                    # Extension sur l’"encre" autour (utile pour flèches/numéros)
                    for ew in expand_candidates_px:
                        wx1 = max(0, bx1 - ew); wy1 = max(0, by1 - ew)
                        wx2 = min(W, bx2 + ew); wy2 = min(H, by2 + ew)
                        roi = bw[wy1:wy2, wx1:wx2]
                        ys, xs = np.where(roi > 0)
                        if len(xs) == 0:
                            continue
                        x1 = wx1 + int(xs.min()); y1 = wy1 + int(ys.min())
                        x2 = wx1 + int(xs.max()) + 1; y2 = wy1 + int(ys.max()) + 1

                        # Si ça grossit trop, on refuse (sinon ça "mange" le texte)
                        new_area = (x2 - x1) * (y2 - y1)
                        if new_area <= base_area * max_expand_factor:
                            cand_list.append((ew, x1, y1, x2, y2))

                    for ew, x1, y1, x2, y2 in cand_list:
                        s, ar, ink, smallcc = score_bbox(x1, y1, x2, y2)
                        if best is None or s > best[0]:
                            best = (s, ew, x1, y1, x2, y2, ar, ink, smallcc)

                    if best is None:
                        continue

                    _, ew, x1, y1, x2, y2, ar, ink, smallcc = best
                    crop = img.crop((x1, y1, x2, y2))
                    if crop.width > 1280:
                        ratio = 1280.0 / crop.width
                        crop = crop.resize((1280, int(crop.height * ratio)), Image.Resampling.LANCZOS)

                    name = f"p{page_no:04d}_r_{uid:06d}.webp"
                    crop.save(str(img_dir / name), "WEBP", quality=85, method=6)
                    page_item["blocks"].append({
                        "type": "region",
                        "path": f"{img_dir.name}/{name}",
                        "bbox_px": [x1, y1, x2, y2],
                        "meta": {
                            "expand_px": int(ew),
                            "area_ratio": round(ar, 4),
                            "ink_ratio": round(ink, 4),
                            "small_cc": int(smallcc)
                        }
                    })
                    uid += 1

            # B Secours : page vectorielle (pas de bitmap), on tente via l’image rendue
            if (not page_item["blocks"]) and pixel_fallback:
                if text_len >= text_gate_chars:
                    if save_text_pages:
                        name = f"p{page_no:04d}_text.webp"
                        img.save(str(img_dir / name), "WEBP", quality=85, method=6)
                        page_item["blocks"].append({"type": "text_page", "path": f"{img_dir.name}/{name}"})
                else:
                    bw2 = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8), iterations=2)
                    nlabels, labels, stats, _ = cv2.connectedComponentsWithStats(bw2, connectivity=8)
                    if nlabels > 1:
                        # Garde surtout les gros composants (évite les petits caractères)
                        keep = []
                        for k in range(1, nlabels):
                            x, y, w0, h0, a = stats[k]
                            if a >= 1200 or h0 >= 90 or w0 >= 240:
                                keep.append(k)
                        if keep:
                            ys, xs = np.where(np.isin(labels, keep))
                            x1, y1 = int(xs.min()), int(ys.min())
                            x2, y2 = int(xs.max()) + 1, int(ys.max()) + 1
                            mpx = int(margin_pt * zoom)
                            x1 = max(0, x1 - mpx); y1 = max(0, y1 - mpx)
                            x2 = min(W, x2 + mpx); y2 = min(H, y2 + mpx)

                            s, ar, ink, smallcc = score_bbox(x1, y1, x2, y2)
                            if ar <= huge_region_area:
                                crop = img.crop((x1, y1, x2, y2))
                                name = f"p{page_no:04d}_fig_{uid:06d}.webp"
                                crop.save(str(img_dir / name), "WEBP", quality=85, method=6)
                                page_item["blocks"].append({
                                    "type": "figure",
                                    "path": f"{img_dir.name}/{name}",
                                    "bbox_px": [x1, y1, x2, y2],
                                    "meta": {"area_ratio": round(ar,4), "ink_ratio": round(ink,4), "small_cc": int(smallcc)}
                                })
                                uid += 1

            # C Si rien trouvé : option page entière
            if (not page_item["blocks"]) and save_full_page_if_empty:
                name = f"p{page_no:04d}_page.webp"
                img.save(str(img_dir / name), "WEBP", quality=85, method=6)
                page_item["blocks"].append({"type": "page_render", "path": f"{img_dir.name}/{name}"})

            pages_out.append(page_item)

        # JSON par document
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(pages_out, f, ensure_ascii=False, indent=2)

        index.append({
            "pdf": str(pdf_path),
            "out_json": str(out_json),
            "image_dir": str(img_dir),
            "pages": total
        })

    # Index global
    with open(out_root / "index.json", "w", encoding="utf-8") as f:
        json.dump(index, f, ensure_ascii=False, indent=2)

    print("OK:", len(index), "pdf(s) ->", str(out_root))

In [28]:
build_packs_for_path(
    input_path="french---mill-operator's-manual---2023.pdf",
    out_root="packs_out",
    recursive=False,
    expand_candidates_px=(60,120,200),
    max_expand_factor=1.6,
    huge_region_area=0.65,
    area_pen_mult=14.0,
    text_pen_mult=0.55
)

OK: 1 pdf(s) -> packs_out


input_path="..."：le PDF (ou un dossier) à traiter.

out_root="packs_out"：le dossier de sortie (JSON + images).

recursive=False：si input_path est un dossier, ne pas descendre dans les sous-dossiers (mettre True pour tout parcourir).

expand_candidates_px=(60,120,200)：tailles des “fenêtres” (en pixels) pour élargir le cadre autour d’une image et récupérer ce qui est juste à côté (flèches, numéros). Plus c’est petit, plus la découpe reste serrée sur la capture.

max_expand_factor=1.6：limite l’agrandissement max du cadre après extension (ici 1,6× la zone de base). Plus c’est bas, moins on “avale” le texte autour.

huge_region_area=0.65：si la zone dépasse ~65% de la page, elle est pénalisée (pour éviter les gros blocs).

area_pen_mult=14.0：force de la pénalité liée à la taille de la zone. Plus c’est haut, plus on favorise des cadres petits.

text_pen_mult=0.55：force de la pénalité “ça ressemble à du texte” (beaucoup de petits composants). Plus c’est haut, moins le cadre va englober des paragraphes.